# EcoCrop GEE - 작물 기후 적합성 평가

Google Earth Engine을 활용한 전 세계 작물 적합성 분석 도구

## 기능
- FAO EcoCrop 데이터베이스 635종 작물 지원
- 사용자 정의 작물 파라미터 설정 가능
- 전 세계 어디든 분석 가능 (좌표, 바운딩박스, 국가명)
- ERA5, TerraClimate, WorldClim 기후 데이터 지원

## 1. 설치 및 초기화

In [ ]:
# 필요한 패키지 설치 (최초 1회)
# !pip install earthengine-api pandas numpy matplotlib folium geemap

In [ ]:
# 모듈 임포트
from ecocrop_gee import (
    EcoCropGEE,
    create_custom_crop_parameters,
    load_ecocrop_database,
    get_crop_parameters
)
import pandas as pd

## 2. EcoCrop 데이터베이스 탐색

In [ ]:
# 데이터베이스 로드
db = load_ecocrop_database('EcoCrop_DB_secondtrim.csv')

# 사용 가능한 작물 목록 확인
print("총 작물 수:", len(db))
print("\n처음 20개 작물:")
db[['ScientificName', 'COMNAME', 'TOPMN', 'TOPMX', 'ROPMN', 'ROPMX']].head(20)

In [ ]:
# 특정 작물 검색 (예: rice, wheat, corn 등)
search_term = 'rice'
mask = db['COMNAME'].str.lower().str.contains(search_term, na=False)
db[mask][['ScientificName', 'COMNAME', 'TOPMN', 'TOPMX', 'TMIN', 'TMAX', 'ROPMN', 'ROPMX']]

## 3. 기본 분석 예제 - 한국 밀 적합성

In [ ]:
# EcoCropGEE 초기화 (GEE 프로젝트 ID가 필요할 수 있음)
# analyzer = EcoCropGEE(project='your-project-id')
analyzer = EcoCropGEE()

In [ ]:
# 데이터베이스 로드 및 작물 선택
analyzer.load_database('EcoCrop_DB_secondtrim.csv')
analyzer.set_crop('wheat')  # 밀

In [ ]:
# 분석 지역 설정 - 서울 중심 100km 반경
analyzer.set_roi_point(lon=127.0, lat=37.5, buffer_km=100)

In [ ]:
# 기후 데이터 가져오기 (WorldClim - 1970-2000 평균 기후)
analyzer.fetch_climate(source='worldclim')

In [ ]:
# 적합성 점수 계산
analyzer.calculate_suitability(method='annual')

In [ ]:
# 지역 통계 확인
stats = analyzer.get_statistics()

In [ ]:
# 인터랙티브 지도 생성
m = analyzer.create_map()
m

## 4. 다양한 지역 설정 방법

In [ ]:
# 방법 1: 좌표 + 반경
analyzer.set_roi_point(lon=127.0, lat=37.5, buffer_km=50)

# 방법 2: 바운딩 박스 (남서 코너, 북동 코너)
# analyzer.set_roi_bbox(min_lon=125.0, min_lat=33.0, max_lon=130.0, max_lat=38.5)

# 방법 3: 국가명
# analyzer.set_roi_country('South Korea')

## 5. 다양한 기후 데이터 소스

In [ ]:
# 소스 1: WorldClim (1970-2000 기후 평균, 날짜 불필요)
analyzer.fetch_climate(source='worldclim')

# 소스 2: ERA5 (특정 연도, 날짜 필요)
# analyzer.fetch_climate(start_date='2020-01-01', end_date='2020-12-31', source='era5')

# 소스 3: TerraClimate (특정 연도, 더 높은 해상도)
# analyzer.fetch_climate(start_date='2020-01-01', end_date='2020-12-31', source='terraclimate')

## 6. 사용자 정의 작물 파라미터

In [ ]:
# 직접 작물 파라미터 정의
custom_rice = create_custom_crop_parameters(
    name="한국 재래종 벼",
    topmn=22,      # 최적 온도 최소 (°C)
    topmx=30,      # 최적 온도 최대 (°C)
    tmin=15,       # 허용 온도 최소 (°C)
    tmax=38,       # 허용 온도 최대 (°C)
    ropmn=1000,    # 최적 강수량 최소 (mm/년)
    ropmx=2000,    # 최적 강수량 최대 (mm/년)
    rmin=600,      # 허용 강수량 최소 (mm/년)
    rmax=3000,     # 허용 강수량 최대 (mm/년)
    gmin=120,      # 생육 기간 최소 (일)
    gmax=180,      # 생육 기간 최대 (일)
    ktmp=10        # 동해 온도 (°C)
)

print(custom_rice)

In [ ]:
# 사용자 정의 작물로 분석
analyzer2 = EcoCropGEE()
analyzer2.set_crop(custom_params=custom_rice)
analyzer2.set_roi_point(lon=127.0, lat=35.0, buffer_km=50)
analyzer2.fetch_climate(source='worldclim')
analyzer2.calculate_suitability(method='annual')
stats2 = analyzer2.get_statistics()

## 7. 결과 내보내기

In [ ]:
# HTML 지도로 저장
m = analyzer.create_map()
if m:
    m.save('wheat_korea_suitability.html')
    print("지도가 저장되었습니다: wheat_korea_suitability.html")

In [ ]:
# Google Drive로 GeoTIFF 내보내기 (시간이 걸릴 수 있음)
# analyzer.export(
#     filename='wheat_korea_2020',
#     folder='EcoCrop_GEE',
#     scale=1000  # 1km 해상도
# )

## 8. 여러 작물 비교 분석

In [ ]:
# 여러 작물에 대해 같은 지역 분석
crops_to_analyze = ['wheat', 'rice', 'maize', 'soybean', 'potato']
results = {}

for crop_name in crops_to_analyze:
    try:
        analyzer = EcoCropGEE()
        analyzer.load_database('EcoCrop_DB_secondtrim.csv')
        analyzer.set_crop(crop_name)
        analyzer.set_roi_point(lon=127.0, lat=37.5, buffer_km=100)
        analyzer.fetch_climate(source='worldclim')
        analyzer.calculate_suitability(method='annual')
        stats = analyzer.get_statistics()
        results[crop_name] = stats.get('overall_score_mean', 0)
        print(f"{crop_name}: {results[crop_name]:.1f}점")
    except Exception as e:
        print(f"{crop_name}: 분석 실패 - {e}")

In [ ]:
# 결과 시각화
import matplotlib.pyplot as plt

if results:
    plt.figure(figsize=(10, 6))
    plt.bar(results.keys(), results.values(), color='steelblue')
    plt.xlabel('작물')
    plt.ylabel('적합성 점수 (평균)')
    plt.title('서울 지역 작물별 기후 적합성 비교')
    plt.ylim(0, 100)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 9. 파라미터 설명

| 파라미터 | 설명 | 단위 |
|----------|------|------|
| TOPMN | 최적 온도 최소값 | °C |
| TOPMX | 최적 온도 최대값 | °C |
| TMIN | 허용 온도 최소값 (절대 하한) | °C |
| TMAX | 허용 온도 최대값 (절대 상한) | °C |
| KTMP | 동해 온도 (작물 치사 온도) | °C |
| ROPMN | 최적 강수량 최소값 | mm/년 |
| ROPMX | 최적 강수량 최대값 | mm/년 |
| RMIN | 허용 강수량 최소값 | mm/년 |
| RMAX | 허용 강수량 최대값 | mm/년 |
| GMIN | 최소 생육 기간 | 일 |
| GMAX | 최대 생육 기간 | 일 |

### 점수 계산 방식

```
온도/강수량 점수:
- 최적 범위 내: 100점
- 허용 범위 내: 0-100점 (선형 보간)
- 범위 밖: 0점

종합 점수 = min(온도 점수, 강수량 점수)
(제한 요인 방식)
```